# CVE Risk Classification — 05: Contrast Layer (Mythos Stub)
**Imperial Business School Capstone | MSc Cybersecurity + AI/ML**

## Purpose
This notebook implements the **Contrast Layer** — the architectural bridge
between the current hybrid model and the future Claude Mythos validation
oracle.

## Architecture
```
Your Hybrid Model Output
  (LOW / MEDIUM / HIGH / CRITICAL)
          ↓
  MythosContrastLayer
          ↓
   Mythos Available?  ──YES──→  Glasswing API call
          │                      → Disagreement Report
          NO                     → Human Analyst Alert
          ↓
   Store classification in queue
   Return waiting message
   Queue grows as retrospective validation set
   (Ready the moment Mythos becomes accessible)
```

## Research value of the queue
Every CVE classified while waiting becomes a **labelled validation example**.
When Mythos access is granted, running retrospective contrast on the full
queue produces a **Mythos Agreement Rate** — effectively the accuracy of
the hybrid model validated against the most capable cybersecurity AI
ever built. That metric alone could be a publishable finding.

> *Status: Mythos restricted under Project Glasswing (April 2026).*  
> *Access expected: September 2026 (speculative, not confirmed by Anthropic).*

In [2]:
import pandas as pd
import numpy as np
import joblib
import json
import os
from datetime import datetime
from pathlib import Path

os.makedirs('queue', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# ── Load saved model components from notebook 03
knn         = joblib.load('models/knn_clusterer.pkl')
scaler_knn  = joblib.load('models/scaler_knn.pkl')   # ← updated name
tree        = joblib.load('models/hybrid_tree.pkl')
le_target   = joblib.load('models/label_encoder.pkl')
feat_enc    = joblib.load('models/feature_encoders.pkl')
knn_features = joblib.load('models/knn_features.pkl') # ← new — needed for inference


print('Models loaded ✓')

Models loaded ✓


## 1. MythosContrastLayer — Core Class

In [3]:
class MythosContrastLayer:
    """
    Contrast Layer between the hybrid CVE classifier and Claude Mythos.

    When Mythos is available (via Glasswing API):
      - Re-analyses the same CVE independently
      - Flags disagreements with the hybrid model
      - Triggers human analyst review on CRITICAL disagreements

    When Mythos is NOT available (current state):
      - Stores the classification in a persistent queue
      - Returns a structured waiting message
      - Queue accumulates as future retrospective validation set
    """

    SEVERITY_ORDER = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2, 'CRITICAL': 3}

    def __init__(self, mythos_available: bool = False, queue_path: str = 'queue/pending.jsonl'):
        self.mythos_available = mythos_available
        self.queue_path = queue_path

    # ──────────────────────────────────────────────────────────────────────────
    # PUBLIC API
    # ──────────────────────────────────────────────────────────────────────────

    def contrast(self, cve_id: str, your_classification: str,
                 cve_metadata: dict = None) -> dict:
        """
        Main entry point. Call with a CVE-ID and your model's classification.
        Returns a structured result dict.
        """
        if self.mythos_available:
            return self._call_mythos(cve_id, your_classification, cve_metadata)
        else:
            return self._queue_and_wait(cve_id, your_classification, cve_metadata)

    def queue_summary(self) -> pd.DataFrame:
        """Returns a DataFrame of all queued classifications."""
        if not Path(self.queue_path).exists():
            return pd.DataFrame()
        records = []
        with open(self.queue_path, 'r') as f:
            for line in f:
                records.append(json.loads(line.strip()))
        return pd.DataFrame(records)

    def run_retrospective(self, mythos_results: list) -> pd.DataFrame:
        """
        Call this once Mythos access is granted.
        mythos_results: list of dicts with keys cve_id, mythos_classification
        Returns disagreement analysis with agreement rate (Mythos Accuracy Proxy).
        """
        queue_df = self.queue_summary()
        if queue_df.empty:
            print('Queue is empty — nothing to analyse.')
            return pd.DataFrame()

        mythos_df = pd.DataFrame(mythos_results)
        merged = queue_df.merge(mythos_df, on='cve_id', how='inner')
        merged['agrees'] = merged['your_classification'] == merged['mythos_classification']

        # Severity gap (how far off is your model when it disagrees?)
        merged['your_rank']   = merged['your_classification'].map(self.SEVERITY_ORDER)
        merged['mythos_rank'] = merged['mythos_classification'].map(self.SEVERITY_ORDER)
        merged['severity_gap'] = (merged['mythos_rank'] - merged['your_rank']).abs()

        agreement_rate = merged['agrees'].mean()
        print(f'\n=== RETROSPECTIVE MYTHOS CONTRAST REPORT ===')
        print(f'CVEs analysed     : {len(merged)}')
        print(f'Agreement rate    : {agreement_rate:.2%}  ← Mythos Accuracy Proxy')
        print(f'Mean severity gap : {merged["severity_gap"].mean():.2f} (0=exact, 3=max)')
        print(f'Critical misses   : {len(merged[(merged["mythos_classification"]=="CRITICAL") & (merged["your_classification"]!="CRITICAL")])} cases where Mythos said CRITICAL but model did not')
        return merged

    # ──────────────────────────────────────────────────────────────────────────
    # PRIVATE METHODS
    # ──────────────────────────────────────────────────────────────────────────

    def _queue_and_wait(self, cve_id, classification, metadata):
        record = {
            'cve_id'            : cve_id,
            'your_classification': classification,
            'metadata'          : metadata or {},
            'queued_at'         : datetime.utcnow().isoformat(),
            'mythos_status'     : 'pending'
        }
        # Append to JSONL queue (one record per line)
        with open(self.queue_path, 'a') as f:
            f.write(json.dumps(record) + '\n')

        return {
            'status'             : 'queued',
            'cve_id'             : cve_id,
            'your_classification': classification,
            'mythos_assessment'  : None,
            'disagreement'       : None,
            'message'            : (
                f'⏳ [{cve_id}] Classified as {classification} by hybrid model. '
                f'Claude Mythos contrast not yet available — restricted under '
                f'Project Glasswing (April 2026). Classification stored in '
                f'retrospective validation queue. '
                f'Follow: anthropic.com/research for access updates.'
            ),
            'queued_at'          : record['queued_at'],
            'queue_size'         : self._queue_size()
        }

    def _call_mythos(self, cve_id, classification, metadata):
        """
        FUTURE IMPLEMENTATION — Glasswing API call goes here.

        Expected Mythos response structure:
        {
            'cve_id': str,
            'severity': 'LOW'|'MEDIUM'|'HIGH'|'CRITICAL',
            'logic_flaws_detected': [str],
            'algorithmic_blindspots': [str],
            'confidence': float
        }
        """
        # ── Placeholder until Glasswing access is granted ─────────────────────
        raise NotImplementedError(
            'Mythos API not yet accessible. '
            'Set mythos_available=False to use queue mode.'
        )
        # ── Future implementation skeleton:
        # import anthropic
        # client = anthropic.Anthropic(api_key=GLASSWING_API_KEY)
        # response = client.messages.create(
        #     model='claude-mythos-glasswing',
        #     max_tokens=1024,
        #     system='You are a cybersecurity vulnerability analyst...',
        #     messages=[{'role': 'user', 'content': f'Analyse CVE: {cve_id}\n{metadata}'}]
        # )
        # return self._build_disagreement_report(cve_id, classification, response)

    def _queue_size(self):
        if not Path(self.queue_path).exists(): return 0
        with open(self.queue_path) as f:
            return sum(1 for _ in f)


# ── Instantiate (stub mode — Mythos not yet available)
contrast_layer = MythosContrastLayer(mythos_available=False)
print('MythosContrastLayer initialised in STUB mode ✓')

MythosContrastLayer initialised in STUB mode ✓


## 2. Full Inference Pipeline — Model + Contrast Layer

In [10]:
def classify_cve(cve_features: dict) -> dict:
    
    cve_id = cve_features.pop('cve_id', 'UNKNOWN')

    # ── Step 1: Encode categoricals using stored encoders
    encoded = {}
    for col, val in cve_features.items():
        if col in feat_enc:
            try:
                encoded[col] = feat_enc[col].transform([str(val)])[0]
            except ValueError:
                encoded[col] = 0
        else:
            encoded[col] = val

    X = pd.DataFrame([encoded])

    # ── Step 2: Drop leakage columns (not seen during training)
    leakage_cols = ['base_score', 'impact_score', 'exploitability_score',
                    'base_severity', 'published_date', 'description_data', 'cpe_data']
    X = X.drop(columns=[c for c in leakage_cols if c in X.columns])

    # ── Step 3: KNN cluster ID on behavioural features only
    X_knn    = X[knn_features] if all(f in X.columns for f in knn_features) else X
    X_scaled = scaler_knn.transform(X_knn)
    cluster_id = knn.predict(X_scaled)[0]
    X['cluster_id'] = cluster_id

    # ── Step 4: Decision Tree classification
    pred_numeric = tree.predict(X)[0]
    severity = le_target.inverse_transform([pred_numeric])[0]

    # ── Step 5: Contrast layer
    contrast_result = contrast_layer.contrast(
        cve_id=cve_id,
        your_classification=severity,
        cve_metadata={'cluster_id': int(cluster_id), **cve_features}
    )

    return {
        'cve_id'        : cve_id,
        'cluster_id'    : int(cluster_id),
        'classification': severity,
        'contrast'      : contrast_result
    }

## 3. Demo — Classify Sample CVEs

In [11]:
# ── Load a sample of clean data to demo the pipeline
df = pd.read_csv('data/df_clean.csv')
sample = df.sample(5, random_state=99)

print('=== PIPELINE DEMO — 5 SAMPLE CVEs ===\n')
for _, row in sample.iterrows():
    # Build feature dict (excluding non-feature columns)
    skip = ['base_severity', 'description_data', 'cpe_data', 'published_date']
    features = {col: row[col] for col in df.columns if col not in skip}

    try:
        result = classify_cve(features)
        print(f"CVE       : {result['cve_id']}")
        print(f"Cluster   : {result['cluster_id']}")
        print(f"Classified: {result['classification']}")
        print(f"Status    : {result['contrast']['status']}")
        print(f"Message   : {result['contrast']['message']}")
        print('-' * 70)
    except Exception as e:
        print(f"Error on {row.get('cve_id','?')}: {e}")
        print('-' * 70)

=== PIPELINE DEMO — 5 SAMPLE CVEs ===

CVE       : CVE-2023-26058
Cluster   : 2
Classified: MEDIUM
Status    : queued
Message   : ⏳ [CVE-2023-26058] Classified as MEDIUM by hybrid model. Claude Mythos contrast not yet available — restricted under Project Glasswing (April 2026). Classification stored in retrospective validation queue. Follow: anthropic.com/research for access updates.
----------------------------------------------------------------------
CVE       : CVE-2024-9946
Cluster   : 1
Classified: HIGH
Status    : queued
Message   : ⏳ [CVE-2024-9946] Classified as HIGH by hybrid model. Claude Mythos contrast not yet available — restricted under Project Glasswing (April 2026). Classification stored in retrospective validation queue. Follow: anthropic.com/research for access updates.
----------------------------------------------------------------------
CVE       : CVE-2024-42951
Cluster   : 1
Classified: HIGH
Status    : queued
Message   : ⏳ [CVE-2024-42951] Classified as HIGH by

C:\Users\USER\AppData\Local\Temp\ipykernel_11680\1438084719.py:84: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'queued_at'         : datetime.utcnow().isoformat(),
C:\Users\USER\AppData\Local\Temp\ipykernel_11680\1438084719.py:84: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'queued_at'         : datetime.utcnow().isoformat(),
C:\Users\USER\AppData\Local\Temp\ipykernel_11680\1438084719.py:84: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'queued_at'         : datetime.utcnow().isoformat(),
C:\Users\USER\AppData\Loc

## 4. Queue Status Dashboard

In [6]:
queue_df = contrast_layer.queue_summary()

if queue_df.empty:
    print('Queue is empty — run the demo cell above first.')
else:
    print(f'=== MYTHOS VALIDATION QUEUE ===')
    print(f'Total queued CVEs: {len(queue_df)}')
    print(f'Classification distribution:')
    print(queue_df['your_classification'].value_counts())
    print()
    print('Sample queue records:')
    print(queue_df[['cve_id','your_classification','queued_at']].head(10))

    # ── Visualise queue composition
    fig, ax = plt.subplots(figsize=(7, 4))
    order = ['LOW','MEDIUM','HIGH','CRITICAL']
    colors = {'LOW':'#4ECDC4','MEDIUM':'#FFE66D','HIGH':'#FF6B6B','CRITICAL':'#C0392B'}
    counts = queue_df['your_classification'].value_counts().reindex(order, fill_value=0)
    bars = ax.bar(counts.index, counts.values,
                  color=[colors[c] for c in counts.index])
    ax.set_title('Mythos Validation Queue — Classification Distribution')
    ax.set_ylabel('CVEs awaiting Mythos validation')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(val), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/05_queue_distribution.png', dpi=150)
    plt.show()

Queue is empty — run the demo cell above first.


## 5. Future: Retrospective Validation (Placeholder)

When Mythos access is granted via Project Glasswing, run this cell
with real Mythos classifications to compute the **Mythos Agreement Rate**.

In [7]:
# ── FUTURE CELL — uncomment and populate mythos_results when access is granted
#
# mythos_results = [
#     {'cve_id': 'CVE-2024-XXXX', 'mythos_classification': 'CRITICAL'},
#     {'cve_id': 'CVE-2024-YYYY', 'mythos_classification': 'HIGH'},
#     # ... (batch from Glasswing API)
# ]
#
# retrospective_df = contrast_layer.run_retrospective(mythos_results)
# retrospective_df.to_csv('outputs/05_mythos_retrospective.csv', index=False)
# print('Retrospective analysis saved ✓')

print('⏳ Waiting for Claude Mythos / Project Glasswing access.')
print('   This cell will execute the retrospective contrast when available.')
print(f'   Current queue size: {contrast_layer._queue_size()} CVEs')

⏳ Waiting for Claude Mythos / Project Glasswing access.
   This cell will execute the retrospective contrast when available.
   Current queue size: 0 CVEs
